# Swarm-MeZO — Day 4 control: conformity vs loss reward (Colab)

Запускает `scripts/run_reputation.py` на удалённой GPU. Скрипт прогоняет
sweep по обеим веткам репутации:

- **loss** (правило §4): `penalty = |L_i − L_min|` — ключи `beta*`,
  β ∈ {0, 0.1, 0.5, 1, 10};
- **conformity** (контроль, дословное правило лекции, слайд 11):
  `penalty = |L_i − L̄|` — ключи `conf_beta*`, β ∈ {0.1, 0.5, 1, 10}.

Скрипт **идемпотентный** — уже посчитанные ключи в
`outputs/day5_reputation.json` пропускаются. Если loss-ветка (`beta*`)
уже прогнана, реально посчитаются **только 4 новых `conf_beta*`** —
loss-конфиги перекручивать не нужно.

**Время:** ≈1.5ч/конфиг (bf16, L4/A100), ≈3ч на T4. Дозапуск только
conformity-ветки (4 конфига) = 6–12ч; полный sweep с нуля = 13–27ч.

**Перед запуском:** Runtime → Change runtime type → GPU (A100/L4/V100
предпочтительно). Для IID-режима — раскомментировать `SHARDING` в §4.

## 1. Setup: клонирование и зависимости

In [ ]:
!nvidia-smi | head -20

In [ ]:
import os
REPO_DIR = '/content/swarm-mezo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/olegkeatzin/swarm-mezo.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull
%cd $REPO_DIR

In [ ]:
# На Colab ставим зависимости через pip (torch с нужной CUDA уже есть).
!pip install -q 'transformers>=4.40' 'datasets>=2.20' tqdm matplotlib pandas
import torch
print(f'torch {torch.__version__}, cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability(0)
    print(f'compute capability: {cap}  (bf16 fast on cap >= (8,0))')

## 2. Прогрев кеша: модель + датасет

Скачивает RoBERTa-base и SST-2 заранее, чтобы прогон не упирался в сеть.

In [ ]:
from datasets import load_dataset            # ВАЖНО: ДО torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

_ = load_dataset('glue', 'sst2')
_ = AutoTokenizer.from_pretrained('roberta-base')
_ = AutoModelForMaskedLM.from_pretrained('roberta-base')
print('cache OK')

## 3. Санитарный smoke-test (≈1 минута)

10 шагов × N=2 — проверка, что vmap + bf16 + reputation-путь живут на
этой GPU. Покрывает обе ветки (`update_reputations` с `mode`).

In [ ]:
!python scripts/smoke_test_reputation.py 2>&1 | tail -40

## 4. Главный прогон — дозапуск conformity-ветки

Идемпотентно: результаты пишутся в `outputs/day5_reputation.json` после
каждого конфига; перезапуск ячейки продолжит с недосчитанного. Уже
посчитанные `beta*` (loss-ветка) пропускаются автоматически.

Для IID-режима (чистый fitness-сигнал, ключ §4-теории) — раскомментировать
`SHARDING=iid`; результаты тогда лягут в `day5_reputation_iid.json`.

In [ ]:
import os
# os.environ['SHARDING'] = 'iid'   # <- раскомментировать для IID-контроля
!mkdir -p outputs && python -u scripts/run_reputation.py 2>&1 | tee -a outputs/day5_reputation.log

## 5. Быстрый превью — quality vs conformity

Таблица и кривые на лету. Полный анализ — в
`notebooks/07_conformity_control.ipynb`.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

SHARDING = os.environ.get('SHARDING', 'dirichlet')
fname = 'day5_reputation_iid.json' if SHARDING == 'iid' else 'day5_reputation.json'
d = json.loads(open(f'outputs/{fname}').read())

def mode_of(h):
    return h.get('mode', 'loss')

runs = dict(sorted(d['runs'].items(),
                   key=lambda kv: (mode_of(kv[1]), kv[1]['beta'])))
print(f"{len(runs)} configs\n")
print(f"{'run':>14} | {'mode':>10} | {'β':>6} | {'val_acc':>8} | "
      f"{'loss':>7} | {'rep share':>9}")
for name, h in runs.items():
    reps = h.get('reputations') or []
    share = float(np.asarray(reps[-1]).max() / np.sum(reps[-1])) if reps else float('nan')
    print(f"{name:>14} | {mode_of(h):>10} | {h['beta']:>6} | "
          f"{h['eval_acc'][-1]:>8.4f} | {h['eval_loss'][-1]:>7.4f} | {share:>9.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=110)
all_betas = sorted({h['beta'] for h in runs.values()})
cmap = plt.get_cmap('viridis')
bcolor = {b: cmap(i / max(len(all_betas) - 1, 1)) for i, b in enumerate(all_betas)}
for name, h in runs.items():
    is_conf = mode_of(h) == 'conformity'
    ax.plot(h['eval_step'], h['eval_acc'],
            ls='--' if is_conf else '-',
            marker='s' if is_conf else 'o', markersize=3,
            color=bcolor[h['beta']],
            label=f"{'conf' if is_conf else 'loss'} β={h['beta']}")
ax.set_xlabel('per-agent step'); ax.set_ylabel('SST-2 val accuracy')
ax.set_title('Day 4: quality (solid) vs conformity (dashed) reward')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

## 6. Скачивание результатов

Заберите JSON + лог локально для полной визуализации в
`notebooks/07_conformity_control.ipynb`.

In [ ]:
from google.colab import files
files.download(f'outputs/{fname}')
files.download('outputs/day5_reputation.log')

**Альтернатива** — закоммитить результаты прямо из Colab (нужен GitHub
PAT в `userdata`):

```python
from google.colab import userdata
token = userdata.get('GITHUB_PAT')
!cd /content/swarm-mezo && \
  git config user.email 'colab@local' && \
  git config user.name 'Colab Runner' && \
  git add outputs/day5_reputation*.json outputs/day5_reputation.log && \
  git commit -m 'Day 4: conformity-control β sweep' && \
  git push https://$token@github.com/olegkeatzin/swarm-mezo.git main
```